In [5]:
# Ignore
from IPython.display import Image
from threading import Thread, Lock

## Q1. Write pseudocode for a parallel version of merge sort using a serial merge step. Compute the span and parallelism of your code. How useful is this code?

You may assume that the work is unchanged, $O(n\log n)$

In [ ]:
print_lock = Lock()

# def say_hi(name):
#     print(f"Hi from {name}")
        
def say_hi(name, num):
    with print_lock:
        print(f"Hi from {name}")
    return num
    
t = Thread(target=say_hi, args=("Test", 0,))
t1 = Thread(target=say_hi, args=("Test1", 1,))
t2 = Thread(target=say_hi, args=("Test2", 2,))
t.start()
t1.start()
t2.start()

print(t)
print(t1)
print(t2)
# 拿不到返回值的

t.join()
t1.join()
t2.join()
print("Done")


Hi from Test
Hi from Test1
Hi from Test2
<Thread(Thread-147 (say_hi), stopped 6269284352)>
<Thread(Thread-148 (say_hi), stopped 6286110720)>
<Thread(Thread-149 (say_hi), stopped 6302937088)>
Done


In [17]:
def merge_sort_parallel(X, result):
    if len(X) == 0:
        return
    if len(X) == 1:
        result.append(X[0])
        return
    mid = len(X) // 2
    R = []
    L = []
    t = Thread(target=merge_sort_parallel, args=(X[mid:], R,))
    t.start()
    merge_sort_parallel(X[:mid], L)
    t.join()
    i = j = 0
    while i < len(R) or j < len(L):
        if i == len(R):
            result.append(L[j])
            j += 1
            continue
        if j == len(L):
            result.append(R[i])
            i += 1
            continue
        
        if R[i] < L[j]:
            result.append(R[i])
            i += 1
        else:
            result.append(L[j])
            j += 1
    return

result = []
a = [1, 9, 28, 2, 42, 1, 8, 1, 78]
merge_sort_parallel(a, result)
print(result)
    

[1, 1, 1, 2, 8, 9, 28, 42, 78]


### Q1 Analysis

**代码结构**：spawn 一个子线程跑左半边，主线程同时跑右半边，sync 之后做**串行 merge**——符合 Q1 要求（递归并行 + merge 串行）。

#### Work

题目假设：
$$T_1(n) = 2\,T_1(n/2) + O(n) = O(n \log n)$$

（master theorem：$a=b=2,\ d=1,\ \log_b a = 1 = d$，case 2。）

#### Span

关键观察：两个递归调用 **并行执行**，所以 span 在它们之间取 $\max$ 而不是相加。merge 步骤是串行的 $O(n)$。

$$T_\infty(n) = \max\bigl(T_\infty(n/2),\ T_\infty(n/2)\bigr) + O(n) = T_\infty(n/2) + O(n)$$

解递归（master theorem：$a=1,\ b=2,\ d=1,\ \log_b a = 0 < d$，case 3，由 $f(n)$ 主导）：

$$T_\infty(n) = O(n)$$

直观：把递归展开，每一层 merge 的串行成本是 $n,\ n/2,\ n/4,\ \dots$，求和仍是 $O(n)$。

#### Parallelism

$$\text{Parallelism} = \frac{T_1(n)}{T_\infty(n)} = \frac{O(n \log n)}{O(n)} = O(\log n)$$

#### How useful is this code?

**用处有限。** 并行度 $\log n$ 增长极慢：

| $n$        | $\log_2 n$ | 能利用的处理器数 |
|-----------:|:----------:|:----------------:|
| $10^3$     | 10         | ~10              |
| $10^6$     | 20         | ~20              |
| $10^9$     | 30         | ~30              |

即使数据规模到十亿，也只能有效利用约 30 个处理器，远低于现代多核机器的能力。

**瓶颈**：最顶层那次 merge 要在长度 $n$ 的数组上跑 $O(n)$ 时间，**完全串行**——这就是 span 卡在 $O(n)$、parallelism 只能到 $O(\log n)$ 的根源。

**改进方向**：把 merge 步骤也并行化（即 Q2 / Q3 要做的事），把 merge 的 span 从 $O(n)$ 降到 $O(\log^2 n)$，最终能让 parallelism 达到 $O(n / \log^2 n)$，才是真正有意义的并行度。

## Q2. Write pseudocode for a parallel merge step, using the hints below. Compute its span.

If you're merging L and R, find the midpoint value x of L, and divide L and R into halves which are less than or greater than x. Use an efficient (serial) algorithm to find the midpoint of R. Find an upper bound of the span of the two spawned merge steps.

In [2]:
Image(url="parallel-merge-hint.png")


<div class="alert alert-block alert-info">
You may want to use the following variant of the master method:

If $T(n)=aT(n/b)+f(n)$ and $f(n)=\Theta(n^{\log_b a}(\log n)^k)$ then $T(n)=\Theta(n^{\log_b a}(\log n)^{k+1})$.
</div>

## Q3. Write pseudocode for a fully parallel merge sort, and compute its span and parallelism. How useful is this code?

You may again assume the work is $O(n\log n)$.

## Q4 (optional). Write code to implement this and verify that it works.

You don't need to actually use multiple processors, just verify that the code works.